### Alternative to XLOOKUP
df = df.merge(master, on="key", how="left")

### Alternative to SUMIFS
df = df.groupby(["key1", "key2"], as_index=False)["amount"].sum()

| Excel Function                             | Python Alternative         | Practical Recommendation            |
| ------------------------------------------ | -------------------------- | ----------------------------------- |
| `XLOOKUP` for retrieving one column        | `map`                      | Fastest and simplest                |
| `XLOOKUP` for retrieving multiple columns  | `merge`                    | Most practical for real-world use   |
| `SUMIFS` for a single calculation          | `loc + conditions + sum()` | Suitable for one-off calculations   |
| `SUMIFS` for creating a summary table      | `groupby`                  | Most recommended for large datasets |
| `SUMIFS` results returned to original data | `groupby + merge`          | More stable and faster than Excel   |


In [1]:
import pandas as pd

### Alternative to XLOOKUP

In [2]:
# Alternative to XLOOKUP
master = pd.DataFrame({
    "Ticker": ["AAPL", "MSFT", "GOOG"],
    "Sector": ["Tech", "Tech", "Communication"]
})

print("master")
display(master)


df = pd.DataFrame({
    "Ticker": ["AAPL", "MSFT", "GOOG", "TSLA"]
})

print("df:original")
display(df)



df["Sector"] = df["Ticker"].map(
    master.set_index("Ticker")["Sector"]
).fillna("")


print("df: after mapping Sector")
display(df)

master


,Ticker,Sector
0,AAPL,Tech
1,MSFT,Tech
2,GOOG,Communication


df:original


,Ticker
0,AAPL
1,MSFT
2,GOOG
3,TSLA


df: after mapping Sector


,Ticker,Sector
0,AAPL,Tech
1,MSFT,Tech
2,GOOG,Communication
3,TSLA,


In [3]:
master = pd.DataFrame({
    "Ticker": ["AAPL", "MSFT", "GOOG", "Toyota"],
    "Sector": ["Tech", "Tech", "Communication","XX"],
    "Country": ["US", "US", "US","JP"],
    "Currency": ["USD", "USD", "USD","JPY"],
})

print("master")
display(master)


df = pd.DataFrame({
    "Ticker": ["AAPL", "MSFT", "GOOG", "Toyota"]
})

print("df:original")
display(df)


df = df.merge(
    master[["Ticker", "Sector", "Country", "Currency"]],
    on="Ticker",
    how="left"
).fillna("")

print("df: after")
display(df)

master


,Ticker,Sector,Country,Currency
0,AAPL,Tech,US,USD
1,MSFT,Tech,US,USD
2,GOOG,Communication,US,USD
3,Toyota,XX,JP,JPY


df:original


,Ticker
0,AAPL
1,MSFT
2,GOOG
3,Toyota


df: after


,Ticker,Sector,Country,Currency
0,AAPL,Tech,US,USD
1,MSFT,Tech,US,USD
2,GOOG,Communication,US,USD
3,Toyota,XX,JP,JPY


### Alternative to SUMIFS

In [4]:
master = pd.DataFrame({
    "Date": ["2026-06-20", "2026-06-20", "2026-06-21", "2026-06-21"],
    "Product": ["A", "B", "A", "A"],
    "Amount": [100, 200, 300, 400]
})
master["Date"] = pd.to_datetime(master["Date"])

print("master")
display(master)

target_date = "2026-06-21"
target_product = "A"


total = master.loc[
    master["Product"] == target_product,
    "Amount"
].sum()

print("Product:A")
print(total, "\n")


total = master.loc[
    (master["Date"] == target_date) &
    (master["Product"] == target_product),
    "Amount"
].sum()

print(f"{target_date} and Product:{target_product}")
print(total, "\n")

df = master.groupby("Date", as_index=False)["Amount"].sum()
print("by Date")
display(df)

df = master.groupby("Product", as_index=False)["Amount"].sum()
print("by Product")
display(df)

print("by Date, Product")
df = master.groupby(["Date", "Product"], as_index=False)["Amount"].sum()
display(df)

master


,Date,Product,Amount
0,2026-06-20,A,100
1,2026-06-20,B,200
2,2026-06-21,A,300
3,2026-06-21,A,400


Product:A
800 

2026-06-21 and Product:A
700 

by Date


,Date,Amount
0,2026-06-20,300
1,2026-06-21,700


by Product


,Product,Amount
0,A,800
1,B,200


by Date, Product


,Date,Product,Amount
0,2026-06-20,A,100
1,2026-06-20,B,200
2,2026-06-21,A,700


In [5]:
df = pd.DataFrame({
    "Date": ["2026-06-20", "2026-06-20"],
    "Product": ["A", "B"]
})
df["Date"] = pd.to_datetime(df["Date"])
display(df)


df = df.merge(
    master,
    on=["Date", "Product"],
    how="left"
).fillna(0)


display(df)

,Date,Product
0,2026-06-20,A
1,2026-06-20,B


,Date,Product,Amount
0,2026-06-20,A,100
1,2026-06-20,B,200


In [6]:
# Incorrect example
df = pd.DataFrame({
    "Date": ["2026-06-20", "2026-06-20"],
    "Product": ["A", "B"]
})
df["Date"] = pd.to_datetime(df["Date"])
df["Amount"] = 0
print("Initial")
display(df)


df = df.merge(
    df,
    on=["Date", "Product"],
    how="left"
).fillna(0)

print("Incorrect example")
display(df)

Initial


,Date,Product,Amount
0,2026-06-20,A,0
1,2026-06-20,B,0


Incorrect example


,Date,Product,Amount_x,Amount_y
0,2026-06-20,A,0,0
1,2026-06-20,B,0,0


In [7]:
df = pd.DataFrame({
    "Date": ["2026-06-20", "2026-06-20"],
    "Product": ["A", "B"]
})
df["Date"] = pd.to_datetime(df["Date"])

df["Amount"] = 0
print("Initial")
display(df)


summary = (
    master.groupby(["Date", "Product"])["Amount"]
      .sum()
)

df["Amount"] = (
    df.set_index(["Date", "Product"]).index
    .map(summary)
)

df["Amount"] = df["Amount"].fillna(0)
print("After")
display(df)

Initial


,Date,Product,Amount
0,2026-06-20,A,0
1,2026-06-20,B,0


After


,Date,Product,Amount
0,2026-06-20,A,100
1,2026-06-20,B,200


In [8]:
df = (
    master.pivot_table(
        index="Date",
        columns="Product",
        values="Amount",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

display(df)

Product,Date,A,B
0,2026-06-20,100,200
1,2026-06-21,700,0


In [9]:
df = master[["Date"]].copy()
df = df.drop_duplicates(subset=["Date"]).reset_index(drop=True)
L = master["Product"].drop_duplicates().tolist()

for l in L:
    df[l] = df["Date"].map(
        master.loc[master["Product"] == l]
              .groupby("Date")["Amount"]
              .sum()
    ).fillna(0)

df

,Date,A,B
0,2026-06-20,100,200.0
1,2026-06-21,700,0.0
